# DSA 504 — Class 5
## pandas: Select, Filter, Sort, and Group

**Date:** Wednesday, Sep 16
**Reading:** *Python for Data Analysis*, ch. 8, 10
**Today's deliverable:** HW1 is assigned at the end of class (due before Class 6)

---

Today we start actually *asking questions* of `retail_sales.csv` — the same dataset from Class 4. Remember: this data still has the messiness we spotted last class (inconsistent category names, missing values, duplicates). We are **still not cleaning it today** — that's Classes 6–7. Today's tools work fine alongside messy data; we'll just get answers that are a little bit off because of it, which is itself a useful lesson in why cleaning matters.

### Learning goals
By the end of this class, you will be able to:
- Select one or more columns from a DataFrame
- Filter rows using boolean conditions, including combining multiple conditions
- Sort a DataFrame by one or more columns
- Group data and compute aggregate statistics per group
- Chain these operations together to answer real questions of the dataset


In [ ]:
import pandas as pd
import numpy as np

sales = pd.read_csv("retail_sales.csv")
sales.head()


## 1. Selecting Columns

You've already seen single-column selection in Class 4. Today we go further: selecting multiple columns at once, and understanding what type of object each kind of selection returns.


In [ ]:
# Selecting a single column -- returns a Series
revenue_column = sales["revenue"]
print(type(revenue_column))
revenue_column.head()


In [ ]:
# Selecting multiple columns -- pass a LIST of column names, returns a DataFrame
subset = sales[["store", "category", "revenue"]]
print(type(subset))
subset.head()


**Common mistake:** `sales["store", "category"]` (no inner brackets) causes an error — you must pass a list, so it's double brackets: `sales[["store", "category"]]`.


In [ ]:
# This is a common error students hit -- shown here on purpose, safely caught
try:
    broken = sales["store", "category"]
except KeyError as e:
    print(f"Error: {e}")
    print("Fix: use double brackets -- sales[['store', 'category']]")


## 2. Filtering Rows (Boolean Indexing)

Filtering is how you answer questions like "show me only the rows where X." The mechanism is called **boolean indexing**: you write a condition that produces `True`/`False` for every row, and pandas keeps only the `True` ones.


In [ ]:
# Step 1: a condition on its own produces a Series of True/False, one per row
is_high_revenue = sales["revenue"] > 3000
print(is_high_revenue.head())
print(type(is_high_revenue))


In [ ]:
# Step 2: put that condition INSIDE square brackets to filter the DataFrame
high_revenue_sales = sales[sales["revenue"] > 3000]
print(f"Rows before filtering: {len(sales)}")
print(f"Rows after filtering: {len(high_revenue_sales)}")
high_revenue_sales.head()


### Filtering with multiple conditions

Use `&` for AND and `|` for OR — **not** the plain Python keywords `and`/`or`, which don't work correctly on pandas Series. Each condition needs its own parentheses.


In [ ]:
# Multiple conditions with & (AND) -- both must be true
utica_electronics = sales[(sales["store"] == "Utica") & (sales["category"] == "Electronics")]
print(len(utica_electronics))
utica_electronics.head()


In [ ]:
# Multiple conditions with | (OR) -- either can be true
utica_or_rome = sales[(sales["store"] == "Utica") | (sales["store"] == "Rome")]
print(len(utica_or_rome))


### Filtering with `.isin()`

When you want to match against several possible values in one column, `.isin()` is cleaner than chaining multiple `|` conditions.


In [ ]:
# Instead of: (store == "Utica") | (store == "Rome") | (store == "Albany")
selected_stores = sales[sales["store"].isin(["Utica", "Rome", "Albany"])]
print(len(selected_stores))
selected_stores["store"].unique()


### Filtering with `.isna()` and `.notna()`

You already used `.isna().sum()` in Class 4 to *count* missing values. Today, use `.isna()` on its own to actually filter down to just the problem rows — useful for inspecting what's really going on before Classes 6–7 fix it.


In [ ]:
# Rows where units_sold is missing
missing_units = sales[sales["units_sold"].isna()]
print(len(missing_units))
missing_units.head()


## 3. Sorting

`.sort_values()` reorders rows by one or more columns.


In [ ]:
# Sort by a single column, ascending (default)
sales.sort_values("revenue").head()


In [ ]:
# Sort descending -- highest revenue first
sales.sort_values("revenue", ascending=False).head()


In [ ]:
# Sort by multiple columns -- store first, then revenue within each store
sales.sort_values(["store", "revenue"], ascending=[True, False]).head(10)


**Note:** `.sort_values()` returns a *new* sorted DataFrame by default — it doesn't change `sales` itself, the same way `sorted()` worked on lists back in Class 3. If you want to keep the sorted version, assign it to a variable.


## 4. Grouping — Aggregating by Category

`.groupby()` is one of the most useful tools in all of pandas. It answers questions like "what's the total revenue *per store*?" instead of one overall total.


In [ ]:
# Total revenue per store
revenue_by_store = sales.groupby("store")["revenue"].sum()
print(revenue_by_store)
print(type(revenue_by_store))


In [ ]:
# Average units sold per category
avg_units_by_category = sales.groupby("category")["units_sold"].mean()
print(avg_units_by_category)


**Talking point:** notice the category grouping above is split into far more groups than the 5 real categories — this is the `"Electronics"` vs `"electronics"` problem from Class 4 showing up concretely now. Grouping messy category data gives you *misleading* per-category totals, since what should be one group is scattered across several. This is exactly why cleaning happens before serious analysis.


In [ ]:
# Grouping by MULTIPLE columns -- store AND category together
revenue_by_store_category = sales.groupby(["store", "category"])["revenue"].sum()
print(revenue_by_store_category.head(10))


### Multiple aggregations at once with `.agg()`

Often you want more than one statistic per group — `.agg()` lets you request several at once.


In [ ]:
# Multiple statistics per store, in one call
summary = sales.groupby("store")["revenue"].agg(["sum", "mean", "count"])
print(summary)


In [ ]:
# .reset_index() turns the group labels back into a regular column,
# which is often useful before sorting or saving the result
summary_reset = summary.reset_index()
print(summary_reset)
print(type(summary_reset))


## 5. Combining Filter, Group, and Sort

Real analysis usually chains several of today's tools together in one line.


In [ ]:
# Question: which store had the highest total revenue, restricted to just electronics-labeled rows?
electronics_only = sales[sales["category"].str.lower() == "electronics"]
result = (
    electronics_only
    .groupby("store")["revenue"]
    .sum()
    .sort_values(ascending=False)
)
print(result)


**Note the `.str.lower()` trick above** — converting to lowercase before comparing/grouping is a quick (partial) workaround for the category-casing problem, letting `"Electronics"` and `"electronics"` count as the same thing for this one query. It doesn't fix the underlying data — the `ELECTRONICS` variant and the stray `"Cloth ing"` typo, for instance, still need real cleaning — but it's a useful preview of what's coming in Classes 6–7.


---
## Guided Practice

Work through these using `sales`, already loaded above.


### Exercise 1 — Select and filter
Select just the `store`, `date`, and `units_sold` columns, then filter that result down to rows where `units_sold` is greater than 100.


In [ ]:
# Exercise 1 — your code here



### Exercise 2 — Multiple conditions
Filter the dataset to rows where the store is `"Rome"` **and** revenue is greater than 2000. How many rows match?


In [ ]:
# Exercise 2 — your code here



### Exercise 3 — Sort
Sort the dataset by `units_sold` in descending order, and print just the top 5 rows for the `store`, `category`, and `units_sold` columns.


In [ ]:
# Exercise 3 — your code here



### Exercise 4 — Group
Compute total revenue per store. Which store has the highest total?


In [ ]:
# Exercise 4 — your code here



### Exercise 5 (stretch) — Combine everything
Filter to rows where `store` is `"Syracuse"`, group the result by `category`, compute both the sum and count of `revenue` per category using `.agg()`, and sort the result by summed revenue in descending order.


In [ ]:
# Exercise 5 — your code here



---
## Solutions

Try each exercise yourself first. These are here for after class, or once you're stuck.


---
## Wrap-up

**Recap:** selecting one or more columns, filtering rows with boolean conditions (`&`, `|`, `.isin()`, `.isna()`), sorting with `.sort_values()`, and grouping with `.groupby()` plus `.agg()` for multiple statistics at once — and chaining all of these together in a single query.

**Common mistakes to watch for:**
- Using Python's `and`/`or` instead of `&`/`|` on pandas conditions
- Forgetting parentheses around each condition when combining them: `(a) & (b)`, not `a & b`
- Passing a single string instead of a list when selecting multiple columns: `sales[["store", "category"]]`, not `sales["store", "category"]`
- Forgetting that `.sort_values()` and `.groupby()` return *new* objects — they don't modify `sales` in place unless you reassign it

**HW1 is assigned today** — it builds directly on select/filter/sort/group, applied to this same dataset. Due before Class 6.

**Before next class (Sep 21):** Class 6 is Data Cleaning I — we finally fix the missing values and inconsistent categories we've been noticing (and working around) since Class 4.
